# Handling Outliers — The Z-Score Method

This is the first of three outlier-handling notebooks, each suited to a different kind of distribution. This one uses the **Z-score** method (±3 standard deviations from the mean) — valid specifically when the column is roughly **normally distributed**, since the "99.7% of data within 3 std" rule only holds for a normal curve.

## 1. Setup and Data

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_csv('placement.csv')

FileNotFoundError: [Errno 2] No such file or directory: 'placement.csv'

In [ ]:
df.shape

In [ ]:
df.sample(5)

## 2. Checking the Shape First

The Z-score method is only appropriate for a column that's close to normal — checking that assumption before applying it matters.

*(`sns.distplot()` is removed on current seaborn — replaced with `sns.histplot(kde=True)`.)*

In [ ]:
plt.figure(figsize=(16, 5))
plt.subplot(1, 2, 1)
sns.histplot(df['cgpa'], kde=True)
plt.title('cgpa')

plt.subplot(1, 2, 2)
sns.histplot(df['placement_exam_marks'], kde=True)
plt.title('placement_exam_marks')

plt.show()


In [ ]:
df['placement_exam_marks'].skew()

`cgpa` looks close to a normal bell curve — a good fit for the Z-score method, used below. `placement_exam_marks` has a much higher skew value — clearly **not** normal, which is exactly why it gets a different method (IQR) in the next notebook instead.

## 3. Computing the Z-Score Boundaries

In [ ]:
print("Mean value of cgpa",df['cgpa'].mean())
print("Std value of cgpa",df['cgpa'].std())
print("Min value of cgpa",df['cgpa'].min())
print("Max value of cgpa",df['cgpa'].max())

The classic rule: values beyond **mean ± 3 standard deviations** are flagged as outliers — for a true normal distribution, about 99.7% of values fall inside that range, so anything outside it is rare enough to call unusual.

In [ ]:
# Finding the boundary values
print("Highest allowed",df['cgpa'].mean() + 3*df['cgpa'].std())
print("Lowest allowed",df['cgpa'].mean() - 3*df['cgpa'].std())

In [ ]:
# Finding the outliers
df[(df['cgpa'] > 8.80) | (df['cgpa'] < 5.11)]

## 4. Approach 1 — Trimming (Removing the Outliers)

## Trimming

In [ ]:
# Trimming

new_df = df[(df['cgpa'] < 8.80) & (df['cgpa'] > 5.11)]
new_df

In [ ]:
print('Rows before trimming:', df.shape[0])
print('Rows after trimming: ', new_df.shape[0])


## 5. Approach 2 — The Same Idea via a Z-Score Column

Rather than comparing directly to the raw cutoff values, converting the whole column to Z-scores first (`(x - mean) / std`) makes "outlier" just mean "Z-score beyond ±3" — equivalent to Approach 1, but the ±3 threshold stays valid even if `cgpa`'s mean/std changes later (no hardcoded cutoff values to keep in sync).

In [ ]:
# Approach 2

# Calculating the Zscore

df['cgpa_zscore'] = (df['cgpa'] - df['cgpa'].mean())/df['cgpa'].std()

In [ ]:
df.head()

In [ ]:
df[df['cgpa_zscore'] > 3]

In [ ]:
df[df['cgpa_zscore'] < -3]

In [ ]:
df[(df['cgpa_zscore'] > 3) | (df['cgpa_zscore'] < -3)]

In [ ]:
# Trimming 
new_df = df[(df['cgpa_zscore'] < 3) & (df['cgpa_zscore'] > -3)]

In [ ]:
new_df

## 6. Approach 3 — Capping (Winsorization) Instead of Trimming

Trimming **removes** rows entirely — costly if there are many outliers, and it shrinks the dataset. **Capping** keeps every row, just pulling extreme values in to sit exactly at the boundary instead of beyond it.

## Capping

In [ ]:
upper_limit = df['cgpa'].mean() + 3*df['cgpa'].std()
lower_limit = df['cgpa'].mean() - 3*df['cgpa'].std()

In [ ]:
lower_limit

In [ ]:
df['cgpa'] = np.where(
    df['cgpa']>upper_limit,
    upper_limit,
    np.where(
        df['cgpa']<lower_limit,
        lower_limit,
        df['cgpa']
    )
)

### Confirming Capping Preserved Every Row

Unlike trimming, the row count should be **completely unchanged** — only the extreme values themselves moved.

In [ ]:
df.shape

In [ ]:
df['cgpa'].describe()

In [ ]:
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
sns.boxplot(x=df['cgpa'])
plt.title('cgpa after capping')

plt.subplot(1, 2, 2)
sns.histplot(df['cgpa'], kde=True)
plt.title('cgpa after capping')
plt.show()


## Summary

| Task | Method |
|---|---|
| Check normality before using Z-score | `sns.histplot(kde=True)` + `.skew()` |
| Compute Z-score boundaries | `mean ± 3*std` |
| Trim (remove outlier rows) | `df[(df[col] < upper) & (df[col] > lower)]` |
| Cap (clip outliers to the boundary) | `np.where(df[col] > upper, upper, np.where(df[col] < lower, lower, df[col]))` |

### Trimming vs. capping
Trimming loses data (fine if outliers are rare and data is plentiful); capping keeps every row but distorts the boundary values slightly. Both require the column to actually be roughly normal for the ±3 std rule to mean anything — for a skewed column, see the IQR method in the next notebook instead.